# Vocoder Deep Dive

This notebook explores the three vocoder types in Vox Machina: **channel**, **phase**, and **LPC**.
Each creates a different style of vocal transformation.

In [ ]:
import numpy as np
from voxmachinae.core.audio_io import AudioBuffer, load_audio
from voxmachinae.synthesis.oscillators import generate_waveform, generate_chord, generate_noise

## Creating Carrier Signals

The carrier determines the *timbre* of the vocoded output. Different carriers produce dramatically different sounds.

In [ ]:
sr = 44100
duration = 3.0

# Simple sawtooth — bright, buzzy (classic vocoder carrier)
saw_carrier = generate_waveform("sawtooth", 110.0, duration, sr)

# Square wave — hollow, clarinet-like
square_carrier = generate_waveform("square", 110.0, duration, sr)

# Chord — multiple notes for richer sound
chord_carrier = generate_chord(
    frequencies=[110.0, 138.59, 164.81],  # A minor triad
    waveform="sawtooth",
    duration=duration,
    sample_rate=sr,
)

# Noise — whispery, breathy effect
noise_carrier = generate_noise("white", duration, sr)

print("Carriers created:")
print(f"  Sawtooth: {saw_carrier.duration:.1f}s")
print(f"  Square:   {square_carrier.duration:.1f}s")
print(f"  Chord:    {chord_carrier.duration:.1f}s")
print(f"  Noise:    {noise_carrier.duration:.1f}s")

## Channel Vocoder

Splits the signal into frequency bands, extracts envelopes from the voice (modulator),
and applies them to the carrier. More bands = more articulate speech.

In [ ]:
from voxmachinae.core.vocoder import ChannelVocoder, ChannelVocoderParams

# Create a synthetic modulator (voice-like signal)
t = np.linspace(0, duration, int(sr * duration), endpoint=False)
modulator_signal = 0.6 * np.sin(2 * np.pi * 220 * t)
modulator_signal += 0.3 * np.sin(2 * np.pi * 440 * t)
modulator_signal += 0.1 * np.sin(2 * np.pi * 660 * t)
# Add amplitude envelope
envelope = np.clip(np.sin(2 * np.pi * 2 * t) * 0.5 + 0.5, 0, 1)
modulator_signal *= envelope
modulator = AudioBuffer(samples=modulator_signal.astype(np.float32), sample_rate=sr)

# 8 bands — lo-fi, robotic
vocoder_8 = ChannelVocoder(ChannelVocoderParams(num_bands=8))
result_8 = vocoder_8.process(modulator, saw_carrier)

# 32 bands — more articulate
vocoder_32 = ChannelVocoder(ChannelVocoderParams(num_bands=32))
result_32 = vocoder_32.process(modulator, saw_carrier)

# 64 bands — very clear speech
vocoder_64 = ChannelVocoder(ChannelVocoderParams(num_bands=64))
result_64 = vocoder_64.process(modulator, saw_carrier)

print("Channel vocoder results:")
print(f"   8 bands: {result_8.duration:.2f}s")
print(f"  32 bands: {result_32.duration:.2f}s")
print(f"  64 bands: {result_64.duration:.2f}s")

## Using Presets

Vox Machina includes presets inspired by iconic vocoder sounds.

In [ ]:
from voxmachinae.presets.vocoder_presets import CHANNEL_VOCODER_PRESETS

for name, preset in CHANNEL_VOCODER_PRESETS.items():
    vocoder = ChannelVocoder(preset)
    result = vocoder.process(modulator, saw_carrier)
    print(f"  {name:20s} -> {result.duration:.2f}s  (bands={preset.num_bands})")

## Phase Vocoder

Uses STFT-based cross-synthesis for spectral manipulation. Can also freeze, stretch, and robotize.

In [ ]:
from voxmachinae.core.vocoder import PhaseVocoder, PhaseVocoderParams

# Cross-synthesis: voice spectral shape + carrier excitation
phase_voc = PhaseVocoder(PhaseVocoderParams(fft_size=2048))
phase_result = phase_voc.process(modulator, saw_carrier)
print(f"Phase vocoder cross-synthesis: {phase_result.duration:.2f}s")

## LPC Vocoder

Linear Predictive Coding — the classic "robot voice" from early speech synthesis.
Extracts formant structure via linear prediction and resynthesizes with a buzz/noise excitation.

In [ ]:
from voxmachinae.core.vocoder import LPCVocoder, LPCVocoderParams

lpc = LPCVocoder(LPCVocoderParams(order=16))
lpc_result = lpc.process(modulator)
print(f"LPC vocoder: {lpc_result.duration:.2f}s (order=16)")

# Higher order = more formant detail
lpc_hi = LPCVocoder(LPCVocoderParams(order=32))
lpc_hi_result = lpc_hi.process(modulator)
print(f"LPC vocoder: {lpc_hi_result.duration:.2f}s (order=32)")

## Summary

| Vocoder Type | Sound Character | Key Parameter | Best For |
|-------------|-----------------|---------------|----------|
| **Channel** | Classic robot/synth voice | `num_bands` (8-64) | Daft Punk, Kraftwerk style |
| **Phase** | Spectral cross-synthesis | `fft_size` | Ethereal, otherworldly |
| **LPC** | Early computer speech | `order` (8-32) | Retro robot voice |